In [2]:
from lightfm import LightFM
from lightfm.datasets import fetch_movielens
from lightfm.evaluation import precision_at_k, auc_score
import pandas as pd
from scipy.sparse import coo_matrix
import numpy as np
from lightfm.data import Dataset
from tqdm import tqdm
from lightfm.cross_validation import random_train_test_split
from sklearn.metrics.pairwise import cosine_similarity
import scipy.sparse as sparse
import metrics
import psycopg2
import yaml
import os
import glob

# Memories

In [3]:
users = pd.read_csv("../../data/raw/users.csv")
posts = pd.read_csv("../../data/raw/posts.csv")

In [397]:
s_p = posts
posts = posts[3:]

### свернуть

In [398]:
def age_to_group(users :pd.DataFrame, col_name: str):
    """Преобразует возраст в определенную группу например 0-18 группа возрастом от 0 до 18
    users: pd.Dataframe - датафрейм пользователей
    col_name: str - название колонки, где хранится возраст
    
    return:
        pd.DataFrame который подавался на вход, добавляет колонку группы и удаляет колонку age
    """
    if col_name in users.columns:
        groups = {"0-11": (0, 11),"12-17": (12, 17), "18-24": (18, 24), "25-34": (25, 34), "35-44": (35, 44), "45-54": (45, 54), "55-64": (55, 64), "64-100": (64, 100)}
        for group in groups:
            l, h = groups[group]
            users.loc[(l <= users[col_name]) & (users[col_name] <= h), col_name + "_group"] = group
        users = users.drop(col_name, axis=1)
        return users
    return users
    

In [399]:
users = age_to_group(users, "age")

In [8]:
posts

,id,user_id,description,location,tag_ids,timestamp
0,26a1ba49-b09e-4275-80c3-e27154ee07d7,2c0dcabb-8260-4b7f-81f0-92913727cada,description,location,"{717125a6-72c4-460e-8b2d-35fdc53671b8,7ead7f8b...",1.713168e+09
1,f7890ff7-c8c2-452a-8362-47371d403c5b,2c0dcabb-8260-4b7f-81f0-92913727cada,description,location,"{717125a6-72c4-460e-8b2d-35fdc53671b8,7ead7f8b...",1.713168e+09
2,6d2cb977-dc1c-4385-a21b-b9db8a6d57db,2c0dcabb-8260-4b7f-81f0-92913727cada,description,location,"{717125a6-72c4-460e-8b2d-35fdc53671b8,f1f73310...",1.713168e+09
3,8afa9da1-8a4c-4c06-9183-cf94d78dfb27,2c0dcabb-8260-4b7f-81f0-92913727cada,description,location,{717125a6-72c4-460e-8b2d-35fdc53671b8},1.713168e+09
4,8819f98e-ba40-41ce-8616-dfd663f94aff,2c0dcabb-8260-4b7f-81f0-92913727cada,dasads,NaN,,1.713437e+09
5,6648d875-e8fb-4ca7-9b56-f5aeed685484,5f0d48eb-d6b9-49a9-b0e7-eba9cd7d5d3a,KewewewAK,eweweewew,"{a87c31c6-ccd0-4a8d-86ca-3768131a211b,92f0a6ca...",1.713517e+09


In [4]:
# def sql_array_to_list()

series_tags_of_posts = posts["tag_ids"].str[1:-1].str.split(",")
series_tags_of_posts
list_str_posts_tags_unique = list(set(np.concatenate(series_tags_of_posts.values).ravel()))
list_str_posts_tags_unique


ValueError: all the input arrays must have same number of dimensions, but the array at index 0 has 1 dimension(s) and the array at index 4 has 0 dimension(s)

In [5]:
series_tags_of_posts

0    [717125a6-72c4-460e-8b2d-35fdc53671b8, 7ead7f8...
1    [717125a6-72c4-460e-8b2d-35fdc53671b8, 7ead7f8...
2    [717125a6-72c4-460e-8b2d-35fdc53671b8, f1f7331...
3               [717125a6-72c4-460e-8b2d-35fdc53671b8]
4                                                  NaN
5    [a87c31c6-ccd0-4a8d-86ca-3768131a211b, 92f0a6c...
Name: tag_ids, dtype: object

In [417]:
posts.values

array([['8afa9da1-8a4c-4c06-9183-cf94d78dfb27',
        '2c0dcabb-8260-4b7f-81f0-92913727cada', 'description',
        'location', None, 1713167640.266993]], dtype=object)

In [401]:
p_userid_unique = posts['user_id'].unique()

In [359]:
tusers = users

In [263]:
# from sklearn.preprocessing import OneHotEncoder
# from scipy.sparse import csr_matrix, vstack



# encoder = OneHotEncoder()
# encoded_data = encoder.fit_transform([[row[1], row[3], row[4], row[5]] for row in tusers.values])
# sparse_user_features = csr_matrix(encoded_data)

In [264]:

# tmp_user = tusers
# for user in tmp_user[['gender', 'city', 'country', 'age_group']].values:
#     add_feature = []
#     for feature_idx in range(len(encoder.categories_)):
#         for type_f in encoder.categories_[feature_idx]:
#             if user[feature_idx] == type_f:
#                 add_feature.append(1)
#             else:
#                 add_feature.append(0)
#     sparse_user_features = vstack([sparse_user_features, csr_matrix(add_feature)])
    
            

In [265]:
# sparse_user_features

### Равернуть

In [392]:
dataset = Dataset()
dataset.fit_partial(users=tusers["id"],
            items=posts["id"],
            item_features=np.concatenate((list_str_posts_tags_unique, p_userid_unique)),
            user_features=np.concatenate((users['gender'].unique(), users['city'].unique(), users['country'].unique(), users['age_group'].unique())))

In [402]:
dataset.fit_partial(users=tusers["id"],
            items=posts["id"])

In [334]:
list_user_features = [(i,[g, c, cn, a]) for i,g,c,cn,a in zip(users['id'], users["gender"], users["city"], users['country'], users['age_group'])]
list_user_features

[('cc384e9c-1bd4-4b61-853b-3833a316341c', ['male', nan, nan, '0-11']),
 ('eea019ca-98a0-4383-a295-8d82676cf00c', ['female', nan, nan, '0-11']),
 ('1df9ec3c-e441-4cda-91c9-67ee73720f05', ['male', nan, nan, '0-11']),
 ('394f1153-43e0-436b-8f79-40272772853a', ['male', nan, nan, '0-11']),
 ('b01be68d-f8b4-4676-9c94-be3b25339e56',
  ['male', 'Russia', 'Krasnoyarsk', '0-11']),
 ('d30b4426-f0cb-44e4-a43c-b21aa63b7c8f',
  ['male', 'Russia', 'Krasnoyarsk', '0-11']),
 ('cf55fb2b-927a-4372-8a4b-caedaf494b5e',
  ['male', 'Russia', 'Krasnoyarsk', '0-11']),
 ('f85d20ce-9d73-4690-8758-e34220617484',
  ['male', 'Russia', 'Krasnoyarsk', '0-11']),
 ('e43b92d6-394c-48cf-92f6-8334221a7268',
  ['male', 'Russia', 'Krasnoyarsk', '0-11']),
 ('f894dbe6-dafd-44a4-a187-8a236ecaa327',
  ['male', 'Russia', 'Krasnoyarsk', '0-11']),
 ('ee4efedc-f5d4-45c9-9e44-0a1e41206f0d',
  ['male', 'Russia', 'Krasnoyarsk', '0-11']),
 ('3c0eb0d5-1e9c-4885-8308-ccce81814201',
  ['male', 'Russia', 'Krasnoyarsk', '0-11']),
 ('7dc5ddd

In [373]:
from sklearn.preprocessing import MultiLabelBinarizer
from scipy.sparse import csr_matrix, hstack
# mlb = MultiLabelBinarizer()
encoded_tags = mlb.transform(series_tags_of_posts)
# mlb_user_id = MultiLabelBinarizer()
encoded_user_id = mlb_user_id.transform(
[[uid] for uid in zip(posts["user_id"])]
)
sm_item_features = hstack(
[csr_matrix(encoded_tags), csr_matrix(encoded_user_id)]
)

/home/dmitry/anaconda3/envs/rec-sys/lib/python3.9/site-packages/sklearn/preprocessing/_label.py:900: UserWarning: unknown class(es) [32] will be ignored
  warnings.warn(


In [419]:
sm_item_features.shape

(4, 4)

In [350]:
# s_sm_item_features = sm_item_features
# sm_item_features = s_sm_item_features

In [374]:
sm_item_features

<1x4 sparse matrix of type '<class 'numpy.int64'>'
	with 1 stored elements in Compressed Sparse Row format>

In [369]:
posts

,id,user_id,description,location,tag_ids,timestamp
3,8afa9da1-8a4c-4c06-9183-cf94d78dfb27,2c0dcabb-8260-4b7f-81f0-92913727cada,description,location,{717125a6-72c4-460e-8b2d-35fdc53671b8},1.713168e+09


In [381]:
print(sm_item_features)

  (0, 0)	1
  (0, 1)	1
  (0, 3)	1
  (1, 0)	1
  (1, 1)	1
  (1, 3)	1
  (2, 0)	1
  (2, 2)	1
  (2, 3)	1
  (3, 3)	1


In [337]:
sm_user_features = dataset.build_user_features(list_user_features)
sm_user_features

<20x27 sparse matrix of type '<class 'numpy.float32'>'
	with 96 stored elements in Compressed Sparse Row format>

In [338]:
list_item_features = [(i, np.concatenate((t, [ui]))) for i,t,ui in zip(posts['id'], series_tags_of_posts, posts['user_id'])]
list_item_features

[('26a1ba49-b09e-4275-80c3-e27154ee07d7',
  array(['717125a6-72c4-460e-8b2d-35fdc53671b8',
         '7ead7f8b-eb31-4bc4-b3b2-442c8c693f36',
         '2c0dcabb-8260-4b7f-81f0-92913727cada'], dtype='<U36')),
 ('f7890ff7-c8c2-452a-8362-47371d403c5b',
  array(['717125a6-72c4-460e-8b2d-35fdc53671b8',
         '7ead7f8b-eb31-4bc4-b3b2-442c8c693f36',
         '2c0dcabb-8260-4b7f-81f0-92913727cada'], dtype='<U36')),
 ('6d2cb977-dc1c-4385-a21b-b9db8a6d57db',
  array(['717125a6-72c4-460e-8b2d-35fdc53671b8',
         'f1f73310-7ba1-433c-914c-4d47fb0948f6',
         '2c0dcabb-8260-4b7f-81f0-92913727cada'], dtype='<U36'))]

In [339]:
sm_item_features = dataset.build_item_features(list_item_features)
sm_item_features

<3x7 sparse matrix of type '<class 'numpy.float32'>'
	with 12 stored elements in Compressed Sparse Row format>

In [403]:
user_id_map, user_feature_map, item_id_map, feature_item_map = dataset.mapping()

In [404]:
LEARNING_RATE = 0.25
NO_EPOCHS = 20
NO_COMPONENTS = 20  # Number of latent factorization
ITEM_ALPHA = 1e-6   # Regularization factor for item features
USER_ALPHA = 1e-6   # Regularization factor for user features

for f in glob.glob("../../data/ratings/ratings_[0-9]*.csv"):
    ratings = pd.read_csv(str(f))
    sm_interactions, sm_weights = dataset.build_interactions(ratings[["user_id","post_id","rating"]][ratings['post_id'] != "8afa9da1-8a4c-4c06-9183-cf94d78dfb27"].values)
    user_id_map, user_feature_map, item_id_map, feature_item_map = dataset.mapping()
    

    
    model.fit_partial(interactions=sm_interactions,
            user_features=sm_user_features,
            item_features=sm_item_features,
            epochs=NO_EPOCHS)

In [422]:
users['id'].values

array(['cc384e9c-1bd4-4b61-853b-3833a316341c',
       'eea019ca-98a0-4383-a295-8d82676cf00c',
       '1df9ec3c-e441-4cda-91c9-67ee73720f05',
       '394f1153-43e0-436b-8f79-40272772853a',
       'b01be68d-f8b4-4676-9c94-be3b25339e56',
       'd30b4426-f0cb-44e4-a43c-b21aa63b7c8f',
       'cf55fb2b-927a-4372-8a4b-caedaf494b5e',
       'f85d20ce-9d73-4690-8758-e34220617484',
       'e43b92d6-394c-48cf-92f6-8334221a7268',
       'f894dbe6-dafd-44a4-a187-8a236ecaa327',
       'ee4efedc-f5d4-45c9-9e44-0a1e41206f0d',
       '3c0eb0d5-1e9c-4885-8308-ccce81814201',
       '7dc5ddde-7724-460d-8d0d-13e994842959',
       '22a6611c-c3d6-41e5-92ee-4d41cefc7e0a',
       '3580ba6b-8b4e-43b5-a88c-2c532994be66',
       '39badfc8-9c69-46e1-8d3a-db45c9ec454d',
       '50f564d5-728d-4a64-96d6-32fd65a3fb91',
       'c9c6e035-74de-4b17-82f5-a2530af4eb17',
       '62112eca-3574-41a4-a547-505a31d126f9',
       '2c0dcabb-8260-4b7f-81f0-92913727cada'], dtype=object)

In [345]:
item_id_map

{'26a1ba49-b09e-4275-80c3-e27154ee07d7': 0,
 'f7890ff7-c8c2-452a-8362-47371d403c5b': 1,
 '6d2cb977-dc1c-4385-a21b-b9db8a6d57db': 2}

In [380]:
model.predict(0, list(item_id_map.values()), user_features=sm_user_features,
            item_features=sm_item_features)

array([-0.0636096, -0.0636096, -0.9151015, -0.6055291], dtype=float32)

In [ ]:
model.pre

<20x4 sparse matrix of type '<class 'numpy.float32'>'
	with 3 stored elements in COOrdinate format>

In [ ]:
sm_interactions

<16x4 sparse matrix of type '<class 'numpy.float32'>'
	with 3 stored elements in COOrdinate format>

In [ ]:
sm_item_features

<4x8 sparse matrix of type '<class 'numpy.float32'>'
	with 15 stored elements in Compressed Sparse Row format>

In [ ]:
sm_interactions

<20x4 sparse matrix of type '<class 'numpy.float32'>'
	with 3 stored elements in COOrdinate format>

# MoviesLen

In [ ]:
ratings = pd.read_csv("../../data/ratings.dat", sep="::", engine="python")
movies = pd.read_csv("../../data/movies.dat", sep="::", engine="python")
users = pd.read_csv("../../data/users.dat", sep="::", engine="python")

In [ ]:
#Удаление не очень активных пользователей
for i in users.UserID:
    if ratings[ratings['UserID'] == i].count()['UserID'] < 100:
        ratings = ratings.drop(ratings[ratings['UserID'] == i].index)

In [ ]:
ratings = ratings.sort_values('Timestamp')

In [ ]:
movies

,MovieID,Title,Genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy
...,...,...,...
3878,3948,Meet the Parents (2000),Comedy
3879,3949,Requiem for a Dream (2000),Drama
3880,3950,Tigerland (2000),Drama
3881,3951,Two Family House (2000),Drama


In [ ]:
TEST_VERY_ACTIVE = 700
TEST_MEDIUM_ACTIVE = 400
TEST_SMALL_ACTIVE = 100

#### Просмотр сколько вообще пользователей посмтрели фильмов определенное количество фильмов

In [ ]:
v_count = 0
m_count = 0
s_count = 0
v_l = []
m_l = []
s_l = []
for i in users.UserID:
    if ratings[ratings['UserID'] == i].count()['UserID'] >= TEST_VERY_ACTIVE:
        v_l.append(i)
        v_count += 1
    elif ratings[ratings['UserID'] == i].count()['UserID'] >= TEST_MEDIUM_ACTIVE:
        m_l.append(i)
        m_count += 1
    elif ratings[ratings['UserID'] == i].count()['UserID'] >= TEST_SMALL_ACTIVE:
        s_l.append(i)
        s_count += 1
print(f"very_c: {v_count} medium_c: {m_count} small_c: {s_count}")

very_c: 177 medium_c: 428 small_c: 2340


In [ ]:
ratings

,UserID,MovieID,Rating,Timestamp
1000138,6040,858,4,956703932
999873,6040,593,5,956703954
1000153,6040,2384,4,956703954
1000192,6040,2019,5,956703977
1000007,6040,1961,4,956703977
...,...,...,...,...
825793,4958,2399,1,1046454338
825438,4958,1407,5,1046454443
825731,4958,2634,3,1046454548
825724,4958,3264,4,1046454548


In [ ]:
def change_ratings_to_valid(c_l: list, ratings: pd.DataFrame):
    # Нужно удалить часть фильмов у пользователей и запонить их, чтобы потом оценить
    val_df = []
    median_user_rating = 0
    for i in c_l:
        for user_id in i:
            base_pref = 0
            tmp_df = []
            len_bigger_then_median = 0
            while (len_bigger_then_median < 10):
                base_pref += 10
                median_user_rating = ratings[ratings['UserID'] == user_id]['Rating'][:-base_pref].median()
                tmp_df = ratings[ratings['UserID'] == user_id][-base_pref:]
                tmp_df = tmp_df[tmp_df['Rating'] >= median_user_rating].values
                len_bigger_then_median = len(tmp_df)
            val_df.append(tmp_df)         
    print(val_df)
            # print(f"median: {median_user_rating}")
            # print(f"vals: {tmp_df}")
    return val_df


In [ ]:
val_df = change_ratings_to_valid([v_l, m_l, s_l], ratings)

[array([[       195,        950,          5, 1040956458],
       [       195,       1090,          5, 1041487792],
       [       195,       3658,          4, 1041488044],
       [       195,       1949,          5, 1041659197],
       [       195,       2587,          4, 1041665302],
       [       195,        632,          5, 1042095314],
       [       195,       3746,          5, 1043562457],
       [       195,       1254,          4, 1044552222],
       [       195,       3007,          4, 1044987990],
       [       195,       1301,          4, 1044988948]]), array([[      216,      3148,         4, 976871432],
       [      216,       198,         3, 976871432],
       [      216,      2991,         3, 976871432],
       [      216,      2502,         4, 976871457],
       [      216,      1475,         3, 976871457],
       [      216,      3107,         4, 976871482],
       [      216,       105,         3, 976871482],
       [      216,       289,         3, 976871503],
   

In [ ]:
len(val_df)

2945

In [ ]:
# ratings_train = pd.read_csv("../../data/ratings_train.csv")

In [ ]:
ratings_train = ratings
for clust in tqdm(val_df):
    for row in clust:
        ratings_train = ratings_train.drop(ratings_train.loc[(ratings_train['UserID'] == row[0]) & (ratings_train['MovieID'] == row[1])].index)
        


100%|██████████| 2945/2945 [22:34<00:00,  2.17it/s]


In [ ]:
# ratings_train.to_csv("../data/ratings_train.csv", index=False)

In [ ]:
users.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6040 entries, 0 to 6039
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   UserID      6040 non-null   int64   
 1   Gender      6040 non-null   category
 2   Age         6040 non-null   category
 3   Occupation  6040 non-null   category
 4   Zip-code    6040 non-null   category
dtypes: category(4), int64(1)
memory usage: 233.9 KB


In [ ]:
for i in users.columns[1:]:
    users[i] = users[i].astype('category')


In [ ]:
users.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6040 entries, 0 to 6039
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   UserID      6040 non-null   int64   
 1   Gender      6040 non-null   category
 2   Age         6040 non-null   category
 3   Occupation  6040 non-null   category
 4   Zip-code    6040 non-null   category
dtypes: category(4), int64(1)
memory usage: 233.9 KB


In [ ]:
users

,UserID,Gender,Age,Occupation,Zip-code
0,1,F,1,10,48067
1,2,M,56,16,70072
2,3,M,25,15,55117
3,4,M,45,7,02460
4,5,M,25,20,55455
...,...,...,...,...,...
6035,6036,F,25,15,32603
6036,6037,F,45,1,76006
6037,6038,F,56,1,14706
6038,6039,F,45,0,01060


In [ ]:
from sklearn.preprocessing import OneHotEncoder
from scipy.sparse import csr_matrix
encoder = OneHotEncoder()
encoded_data = encoder.fit_transform([[row[2], row[3]] for row in users.values])
sparse_user_features = csr_matrix(encoded_data)

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from scipy.sparse import csr_matrix

# Пример DataFrame с данными о фильмах


# Извлекаем текстовые данные для кодирования (в данном случае жанры)
text_data = movies['Genres']

# Инициализируем и обучаем CountVectorizer для кодирования текстовых данных
vectorizer = CountVectorizer(tokenizer=lambda x: x.split('|'))
encoded_data = vectorizer.fit_transform(text_data)

# Преобразуем разреженную матрицу в формат CSR
sparse_movies_features = csr_matrix(encoded_data)

/home/dmitry/anaconda3/envs/rec-sys/lib/python3.9/site-packages/sklearn/feature_extraction/text.py:525: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [ ]:
# users_features = pd.get_dummies(users[['UserID', 'Gender', "Age", "Occupation", "Zip-code"]])

In [ ]:
# users_features

In [ ]:
series_genre_of_movies = movies["Genres"].str.split("|")
list_str_movie_genre_unique = list(set(np.concatenate(series_genre_of_movies).ravel()))

In [ ]:
# user_metadata = users_features.drop(columns=['UserID']).columns

In [ ]:
# user_metadata

In [ ]:
tusers = users[10:]

In [ ]:
dataset = Dataset()
dataset.fit(users=users["UserID"],
            items=movies["MovieID"],
            item_features=list_str_movie_genre_unique,
            user_features=users['Age'].unique().tolist() + users['Occupation'].unique().tolist())

In [ ]:
list_user_features = [(x,[y, z]) for x,y,z in zip(users["UserID"], users["Age"], users['Occupation'])]

In [ ]:
list_user_features

[(1, [1, 10]),
 (2, [56, 16]),
 (3, [25, 15]),
 (4, [45, 7]),
 (5, [25, 20]),
 (6, [50, 9]),
 (7, [35, 1]),
 (8, [25, 12]),
 (9, [25, 17]),
 (10, [35, 1]),
 (11, [25, 1]),
 (12, [25, 12]),
 (13, [45, 1]),
 (14, [35, 0]),
 (15, [25, 7]),
 (16, [35, 0]),
 (17, [50, 1]),
 (18, [18, 3]),
 (19, [1, 10]),
 (20, [25, 14]),
 (21, [18, 16]),
 (22, [18, 15]),
 (23, [35, 0]),
 (24, [25, 7]),
 (25, [18, 4]),
 (26, [25, 7]),
 (27, [25, 11]),
 (28, [25, 1]),
 (29, [35, 7]),
 (30, [35, 7]),
 (31, [56, 7]),
 (32, [25, 0]),
 (33, [45, 3]),
 (34, [18, 0]),
 (35, [45, 1]),
 (36, [25, 3]),
 (37, [25, 9]),
 (38, [18, 4]),
 (39, [18, 4]),
 (40, [45, 0]),
 (41, [18, 4]),
 (42, [25, 8]),
 (43, [25, 12]),
 (44, [45, 17]),
 (45, [45, 16]),
 (46, [18, 19]),
 (47, [18, 4]),
 (48, [25, 4]),
 (49, [18, 12]),
 (50, [25, 2]),
 (51, [1, 10]),
 (52, [18, 4]),
 (53, [25, 0]),
 (54, [50, 1]),
 (55, [35, 12]),
 (56, [35, 20]),
 (57, [18, 19]),
 (58, [25, 2]),
 (59, [50, 1]),
 (60, [50, 1]),
 (61, [25, 17]),
 (62, [35, 3])

In [ ]:
sm_user_features = dataset.build_user_features(list_user_features)
sm_user_features

<6040x6041 sparse matrix of type '<class 'numpy.float32'>'
	with 18102 stored elements in Compressed Sparse Row format>

In [ ]:
list_item_features = [(x,y) for x,y in zip(movies["MovieID"], series_genre_of_movies)]

In [ ]:
list_item_features

[(1, ['Animation', "Children's", 'Comedy']),
 (2, ['Adventure', "Children's", 'Fantasy']),
 (3, ['Comedy', 'Romance']),
 (4, ['Comedy', 'Drama']),
 (5, ['Comedy']),
 (6, ['Action', 'Crime', 'Thriller']),
 (7, ['Comedy', 'Romance']),
 (8, ['Adventure', "Children's"]),
 (9, ['Action']),
 (10, ['Action', 'Adventure', 'Thriller']),
 (11, ['Comedy', 'Drama', 'Romance']),
 (12, ['Comedy', 'Horror']),
 (13, ['Animation', "Children's"]),
 (14, ['Drama']),
 (15, ['Action', 'Adventure', 'Romance']),
 (16, ['Drama', 'Thriller']),
 (17, ['Drama', 'Romance']),
 (18, ['Thriller']),
 (19, ['Comedy']),
 (20, ['Action']),
 (21, ['Action', 'Comedy', 'Drama']),
 (22, ['Crime', 'Drama', 'Thriller']),
 (23, ['Thriller']),
 (24, ['Drama', 'Sci-Fi']),
 (25, ['Drama', 'Romance']),
 (26, ['Drama']),
 (27, ['Drama']),
 (28, ['Romance']),
 (29, ['Adventure', 'Sci-Fi']),
 (30, ['Drama']),
 (31, ['Drama']),
 (32, ['Drama', 'Sci-Fi']),
 (33, ['Adventure', 'Romance']),
 (34, ["Children's", 'Comedy', 'Drama']),
 (35,

In [ ]:
sm_item_features = dataset.build_item_features(list_item_features)
sm_item_features

<3883x3901 sparse matrix of type '<class 'numpy.float32'>'
	with 10291 stored elements in Compressed Sparse Row format>

# 1. Обучение и тест на ratings

In [ ]:
sm_interactions, sm_weights = dataset.build_interactions(ratings[["UserID","MovieID","Rating"]].values)
sm_interactions

KeyboardInterrupt: 

In [ ]:
sm_interactions

<6040x3883 sparse matrix of type '<class 'numpy.int32'>'
	with 847302 stored elements in COOrdinate format>

In [ ]:
user_id_map, user_feature_map, item_id_map, feature_item_map = dataset.mapping()

In [ ]:
sm_train_interactions, sm_test_interactions = random_train_test_split(sm_interactions, test_percentage=0.2, random_state=42)
print(f"Shape of train interactions: {sm_train_interactions.shape}")
print(f"Shape of test interactions: {sm_test_interactions.shape}")

Shape of train interactions: (6040, 3883)
Shape of test interactions: (6040, 3883)


In [ ]:
sm_train_interactions

<6040x3883 sparse matrix of type '<class 'numpy.int32'>'
	with 677841 stored elements in COOrdinate format>

In [ ]:
LEARNING_RATE = 0.25
NO_EPOCHS = 20
NO_COMPONENTS = 20  # Number of latent factorization
ITEM_ALPHA = 1e-6   # Regularization factor for item features
USER_ALPHA = 1e-6   # Regularization factor for user features

model = LightFM(loss="warp",
                no_components=NO_COMPONENTS, 
                learning_rate=LEARNING_RATE, 
                item_alpha=ITEM_ALPHA,
                user_alpha=USER_ALPHA,
                random_state=42)

model.fit_partial(interactions=sm_interactions,
          user_features=sparse_user_features,
          item_features=sm_item_features,
          epochs=NO_EPOCHS)

In [ ]:
model.fit_partial(interactions=sm_interactions,
          user_features=sm_user_features,
          item_features=sm_item_features,
          epochs=NO_EPOCHS)

KeyboardInterrupt: 

## Оценка встроенными методами

In [ ]:
np_arr_prec = precision_at_k(model,
                             test_interactions=sm_test_interactions,
                             user_features=sparse_user_features,
                             item_features=sm_item_features)

In [ ]:
np_arr_prec.mean()

0.14190154

In [ ]:
test_auc = auc_score(model, sm_test_interactions, train_interactions=sm_train_interactions,user_features=sm_user_features,
                             item_features=sm_item_features, num_threads=2).mean()
print('Collaborative filtering test AUC: %s' % test_auc)

Collaborative filtering test AUC: 0.88050944


# 2. Обучение и тест на ratings_train (в котором удалены некоторые фильмы)

In [ ]:
sm_interactions, sm_weights = dataset.build_interactions(ratings_train[["UserID","MovieID","Rating"]].values)
sm_interactions

<6040x3883 sparse matrix of type '<class 'numpy.int32'>'
	with 807967 stored elements in COOrdinate format>

In [ ]:
sm_train_interactions, sm_test_interactions = random_train_test_split(sm_interactions, test_percentage=0.2, random_state=42)

In [ ]:
sparse_user_features

<20x8 sparse matrix of type '<class 'numpy.float64'>'
	with 80 stored elements in Compressed Sparse Row format>

In [ ]:
user_id_map, user_feature_map, item_id_map, feature_item_map = dataset.mapping()

In [ ]:
LEARNING_RATE = 0.25
NO_EPOCHS = 20
NO_COMPONENTS = 20  # Number of latent factorization
ITEM_ALPHA = 1e-6   # Regularization factor for item features
USER_ALPHA = 1e-6   # Regularization factor for user features

model = LightFM(loss="warp",
                no_components=NO_COMPONENTS, 
                learning_rate=LEARNING_RATE, 
                item_alpha=ITEM_ALPHA,
                user_alpha=USER_ALPHA,
                random_state=42)

model.fit(interactions=sm_train_interactions,
          user_features=sparse_user_features,
          item_features=sparse_movies_features,
          epochs=NO_EPOCHS)

### Получение матрицы схожести элементов

In [ ]:
_, np_item_embeddings = model.get_item_representations(features=sparse_movies_features)
print(np_item_embeddings.shape)   # (352, 20)
np_item_embeddings[:2]

(3883, 20)


array([[-3.9402154e-01, -1.2367530e+00, -1.7833997e+00,  1.0649457e+00,
        -6.4317220e-01,  2.3647473e+00, -4.6021447e+00, -2.3143704e+00,
         8.9453125e-01, -2.0062921e+00, -8.2384479e-01,  1.6906636e+00,
         1.7190988e+00, -1.2276769e+00, -6.3924110e-01,  2.2970347e+00,
         1.2670162e+00,  3.1046448e+00,  5.2146148e-04, -1.3436410e+00],
       [ 2.0718361e+01, -6.5617018e+00,  9.1495705e+00, -1.8356833e+01,
         7.9756055e+00, -9.8699923e+00,  1.1630819e+01,  6.3855548e+00,
        -6.8139110e+00,  9.1963997e+00,  1.0331884e+01, -9.4827652e+00,
        -7.7789860e+00,  4.9694796e+00, -2.4231088e-01, -1.1948149e+01,
        -5.8588405e+00, -1.9772484e+01, -2.1357353e+00, -2.7670987e+00]],
      dtype=float32)

In [ ]:
np_item_similarities = cosine_similarity(sparse.csr_matrix(np_item_embeddings)) #сама матрица схожестей
print(np_item_similarities.shape)   # (352, 352)
np_item_similarities[:2]

(3883, 3883)


array([[ 0.9999999 , -0.7437953 ,  0.12326343, ..., -0.2740068 ,
        -0.2740068 , -0.04316745],
       [-0.7437953 ,  0.99999994, -0.00770278, ...,  0.1631929 ,
         0.1631929 , -0.28414106]], dtype=float32)

In [ ]:
df_item_similarities = pd.DataFrame(np_item_similarities)
df_item_similarities.columns = item_id_map.keys()
df_item_similarities.index = item_id_map.keys()
df_item_similarities

,1,2,3,4,5,6,7,8,9,10,...,3943,3944,3945,3946,3947,3948,3949,3950,3951,3952
1,1.000000,-0.743795,0.123263,-0.191892,-0.053089,0.817160,0.123263,0.982228,0.271390,0.629927,...,-0.053089,-0.191892,0.993774,0.180409,0.084428,-0.053089,-0.274007,-0.274007,-0.274007,-0.043167
2,-0.743795,1.000000,-0.007703,-0.243948,-0.373155,-0.989516,-0.007703,-0.617913,0.324863,-0.216175,...,-0.373155,-0.243948,-0.671229,-0.009137,-0.345165,-0.373155,0.163193,0.163193,0.163193,-0.284141
3,0.123263,-0.007703,1.000000,0.225402,0.528720,0.088248,1.000000,0.119477,-0.024009,0.568660,...,0.528720,0.225402,0.101440,0.514335,0.734912,0.528720,-0.458604,-0.458604,-0.458604,0.551490
4,-0.191892,-0.243948,0.225402,1.000000,0.849464,0.213775,0.225402,-0.309948,-0.503542,-0.028799,...,0.849464,1.000000,-0.272917,0.455209,0.637659,0.849464,0.469330,0.469330,0.469330,0.895921
5,-0.053089,-0.373155,0.528720,0.849464,1.000000,0.363707,0.528720,-0.196481,-0.452968,0.212790,...,1.000000,0.849464,-0.147933,0.532649,0.918853,1.000000,-0.067245,-0.067245,-0.067245,0.933132
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3948,-0.053089,-0.373155,0.528720,0.849464,1.000000,0.363707,0.528720,-0.196481,-0.452968,0.212790,...,1.000000,0.849464,-0.147933,0.532649,0.918853,1.000000,-0.067245,-0.067245,-0.067245,0.933132
3949,-0.274007,0.163193,-0.458604,0.469330,-0.067245,-0.204437,-0.458604,-0.257274,-0.194110,-0.410564,...,-0.067245,0.469330,-0.268497,-0.030631,-0.331950,-0.067245,1.000000,1.000000,1.000000,0.132505
3950,-0.274007,0.163193,-0.458604,0.469330,-0.067245,-0.204437,-0.458604,-0.257274,-0.194110,-0.410564,...,-0.067245,0.469330,-0.268497,-0.030631,-0.331950,-0.067245,1.000000,1.000000,1.000000,0.132505
3951,-0.274007,0.163193,-0.458604,0.469330,-0.067245,-0.204437,-0.458604,-0.257274,-0.194110,-0.410564,...,-0.067245,0.469330,-0.268497,-0.030631,-0.331950,-0.067245,1.000000,1.000000,1.000000,0.132505


Суть в том что, для того чтобы брать tp fp нужно будет как-то разграничивать классификацию. Если предсказанный фильм похож на один из фильмов тестовой выборки (выше определенного порога), то это положительный класс. Если он положительный, но не входит в тестовый набор, то это FN, а если входит то будет FP.

#### Нужно получить матрицу в которой будут id пользователя и фильмы, которые он не видел

In [ ]:
SIM_K = 0.7
K_TO_PRED = 100
TAKE_FIRST_PREDICT = 100

In [ ]:
test = []
for clust_idx in range(len(val_df)):
    test.append({"UserID": val_df[clust_idx][0][0], "TestMovies": [], "NoWatched": [], "UserIdMap": {val_df[clust_idx][0][0]: user_id_map[val_df[clust_idx][0][0]]}, "ItemIdMap": {}})
    for row in val_df[clust_idx]:
        test[clust_idx]['TestMovies'].append(row[1])
        test[clust_idx]['ItemIdMap'][row[1]] = item_id_map[row[1]]
test

[{'UserID': 195,
  'TestMovies': [950, 1090, 3658, 1949, 2587, 632, 3746, 1254, 3007, 1301],
  'NoWatched': [],
  'UserIdMap': {195: 194},
  'ItemIdMap': {950: 938,
   1090: 1074,
   3658: 3589,
   1949: 1880,
   2587: 2518,
   632: 627,
   3746: 3677,
   1254: 1234,
   3007: 2938,
   1301: 1281}},
 {'UserID': 216,
  'TestMovies': [3148,
   198,
   2991,
   2502,
   1475,
   3107,
   105,
   289,
   2528,
   1810,
   938,
   647,
   3197],
  'NoWatched': [],
  'UserIdMap': {216: 215},
  'ItemIdMap': {3148: 3079,
   198: 196,
   2991: 2922,
   2502: 2433,
   1475: 1447,
   3107: 3038,
   105: 103,
   289: 286,
   2528: 2459,
   1810: 1746,
   938: 926,
   647: 641,
   3197: 3128}},
 {'UserID': 245,
  'TestMovies': [904, 942, 2206, 1086, 2467, 1625, 931, 1732, 1422, 1018],
  'NoWatched': [],
  'UserIdMap': {245: 244},
  'ItemIdMap': {904: 892,
   942: 930,
   2206: 2137,
   1086: 1070,
   2467: 2398,
   1625: 1582,
   931: 919,
   1732: 1683,
   1422: 1398,
   1018: 1005}},
 {'UserID': 3

In [ ]:
metrics.is_tp(df_item_similarities, [1], 2, 0.8)

False

In [ ]:
# for i in test:
#     i['NoWatched'] = list(set(movies['MovieID'].unique()).difference(set(ratings_train[ratings_train['UserID'] == i['UserID']]["MovieID"].values)))

In [ ]:
test[0]['ItemIdMap']

{950: 938,
 1090: 1074,
 3658: 3589,
 1949: 1880,
 2587: 2518,
 632: 627,
 3746: 3677,
 1254: 1234,
 3007: 2938,
 1301: 1281}

In [ ]:
user_id_map

{1: 0,
 2: 1,
 3: 2,
 4: 3,
 5: 4,
 6: 5,
 7: 6,
 8: 7,
 9: 8,
 10: 9,
 11: 10,
 12: 11,
 13: 12,
 14: 13,
 15: 14,
 16: 15,
 17: 16,
 18: 17,
 19: 18,
 20: 19,
 21: 20,
 22: 21,
 23: 22,
 24: 23,
 25: 24,
 26: 25,
 27: 26,
 28: 27,
 29: 28,
 30: 29,
 31: 30,
 32: 31,
 33: 32,
 34: 33,
 35: 34,
 36: 35,
 37: 36,
 38: 37,
 39: 38,
 40: 39,
 41: 40,
 42: 41,
 43: 42,
 44: 43,
 45: 44,
 46: 45,
 47: 46,
 48: 47,
 49: 48,
 50: 49,
 51: 50,
 52: 51,
 53: 52,
 54: 53,
 55: 54,
 56: 55,
 57: 56,
 58: 57,
 59: 58,
 60: 59,
 61: 60,
 62: 61,
 63: 62,
 64: 63,
 65: 64,
 66: 65,
 67: 66,
 68: 67,
 69: 68,
 70: 69,
 71: 70,
 72: 71,
 73: 72,
 74: 73,
 75: 74,
 76: 75,
 77: 76,
 78: 77,
 79: 78,
 80: 79,
 81: 80,
 82: 81,
 83: 82,
 84: 83,
 85: 84,
 86: 85,
 87: 86,
 88: 87,
 89: 88,
 90: 89,
 91: 90,
 92: 91,
 93: 92,
 94: 93,
 95: 94,
 96: 95,
 97: 96,
 98: 97,
 99: 98,
 100: 99,
 101: 100,
 102: 101,
 103: 102,
 104: 103,
 105: 104,
 106: 105,
 107: 106,
 108: 107,
 109: 108,
 110: 109,
 111: 11

In [ ]:
sm_item_features

<3883x3901 sparse matrix of type '<class 'numpy.float32'>'
	with 10291 stored elements in Compressed Sparse Row format>

In [ ]:
# for i in tqdm(test):
#     list_scores = model.predict(i['UserIdMap'][i["UserID"]], list(item_id_map.values()))
#     series_scores = pd.Series(list_scores)
#     series_scores.index = item_id_map.keys()
#     series_scores.sort_values(ascending=False, inplace=True)
#     predictions = series_scores
#     top_p = []
#     for idx in predictions.index:
#         if ratings_train[(ratings_train['UserID'] == i["UserID"]) & (ratings_train['MovieID'] == idx)]['UserID'].sum():   
#             continue
#         top_p.append(idx)
#         if len(top_p) >= TAKE_FIRST_PREDICT:
#             break
#     tmp_c = 0
#     for j in top_p:
#         if metrics.is_tp(df_item_similarities, i["TestMovies"], j, 0.7):
#             tmp_c += 1
#     i['tp'] = tmp_c
#     i['tn'] = len(top_p) - tmp_c


for i in tqdm(test):
    i['tpIdxs'] = []
    list_scores = model.predict(i['UserIdMap'][i["UserID"]], list(item_id_map.values()), item_features=sparse_movies_features, user_features=sparse_user_features)
    series_scores = pd.Series(list_scores)
    series_scores.index = item_id_map.keys()
    series_scores.sort_values(ascending=False, inplace=True)
    predictions = series_scores
    top_p = []
    for idx in predictions.index:
        if ratings_train[(ratings_train['UserID'] == i["UserID"]) & (ratings_train['MovieID'] == idx)]['UserID'].sum():   
            continue
        top_p.append(idx)
        if len(top_p) >= TAKE_FIRST_PREDICT:
            break
    tmp_c = 0
    for idx, j in enumerate(top_p):
        if metrics.is_tp(df_item_similarities, i["TestMovies"], j, SIM_K):
            tmp_c += 1
            i['tpIdxs'].append(idx + 1)
    i['tp'] = tmp_c
    i['tn'] = len(top_p) - tmp_c

        
        

100%|██████████| 2945/2945 [11:34<00:00,  4.24it/s]


## Оценка

#### Встроенные метрики

In [ ]:
test_auc = auc_score(model, sm_test_interactions, train_interactions=sm_train_interactions,user_features=sparse_user_features,
                             item_features=sparse_movies_features, num_threads=2).mean()
print('Collaborative filtering test AUC: %s' % test_auc)

Collaborative filtering test AUC: 0.65638024


In [ ]:
np_arr_prec = precision_at_k(model,
                             test_interactions=sm_test_interactions,
                             user_features=sparse_user_features,
                             item_features=sparse_movies_features)
mapk = np_arr_prec.mean()
print('Collaborative filtering test mapk: %s' % mapk)

Collaborative filtering test mapk: 0.0567742


#### Кастомные

In [ ]:
sim_acc = []
for i in test:
    sim_acc.append(i['tp'] / TAKE_FIRST_PREDICT)
sim_acc = np.mean(sim_acc)
print('Collaborative filtering test sim acc mean: %s' % sim_acc)


Collaborative filtering test sim acc mean: 0.6921833616298811


In [ ]:
test[0]['tpIdxs']

[14, 16, 25, 35, 44, 45, 46, 51, 56, 59, 69, 78, 79, 85, 89, 94, 98]

In [ ]:
sim_acc_rank = []
for row in test:
    n = 0
    tmp_sum = 0
    for i in range(K_TO_PRED):
        if i + 1 in row['tpIdxs']:
            n += 1
        tmp_sum += n / (i + 1)
    tmp_sum = tmp_sum / K_TO_PRED
    sim_acc_rank.append(tmp_sum)
sim_acc_rank_mean = np.mean(sim_acc_rank)
print('Collaborative filtering test sim acc rank mean: %s' % sim_acc_rank_mean)


Collaborative filtering test sim acc rank mean: 0.7053730553980239


## Популярные фильмы

In [ ]:
ratings_train['MovieID'].value_counts()

MovieID
1196    2083
2858    2003
260     1966
1210    1895
2571    1870
        ... 
3312       1
3621       1
142        1
2742       1
1842       1
Name: count, Length: 3666, dtype: int64

In [ ]:
for i in tqdm(test):
    i['tpIdxs'] = []
    predictions = ratings_train['MovieID'].value_counts()
    top_p = []
    for idx in predictions.index:
        if ratings_train[(ratings_train['UserID'] == i["UserID"]) & (ratings_train['MovieID'] == idx)]['UserID'].sum():   
            continue
        top_p.append(idx)
        if len(top_p) >= TAKE_FIRST_PREDICT:
            break
    tmp_c = 0
    for idx, j in enumerate(top_p):
        if metrics.is_tp(df_item_similarities, i["TestMovies"], j, SIM_K):
            tmp_c += 1
            i['tpIdxs'].append(idx + 1)
    i['tp'] = tmp_c
    i['tn'] = len(top_p) - tmp_c

100%|██████████| 2945/2945 [16:22<00:00,  3.00it/s]


In [ ]:
sim_acc = []
for i in test:
    sim_acc.append(i['tp'] / TAKE_FIRST_PREDICT)
sim_acc = np.mean(sim_acc)
print('Collaborative filtering test sim acc mean: %s' % sim_acc)


Collaborative filtering test sim acc mean: 0.7080882852292021


In [ ]:
sim_acc_rank = []
for row in test:
    n = 0
    tmp_sum = 0
    for i in range(K_TO_PRED):
        if i + 1 in row['tpIdxs']:
            n += 1
        tmp_sum += n / (i + 1)
    tmp_sum = tmp_sum / K_TO_PRED
    sim_acc_rank.append(tmp_sum)
sim_acc_rank_mean = np.mean(sim_acc_rank)
print('Collaborative filtering test sim acc rank mean: %s' % sim_acc_rank_mean)

Collaborative filtering test sim acc rank mean: 0.7124445347026322
